# Publish Triangle Mesh from Multipatch Shapefile

This notebook shows how you can sign in and publish a Triangle Mesh geoscience object from a shapefile containing only multipatch shapes to your chosen Evo workspace.

**Important:** This notebook requires Python 3.10, 3.11, or 3.12 (not 3.14+) due to evo.notebooks dependencies.

In the first cell we create a ServiceManagerWidget which will open a browser window and ask you to sign in.

Once signed in, a widget will be displayed to allow you to select your organisation and an Evo workspace.

__Required:__ In Cell 2, replace `"your-client-id"` with your Evo app client ID before running the cell.

In [ ]:
from evo.notebooks import ServiceManagerWidget

manager = await ServiceManagerWidget.with_auth_code(client_id="your-client-id").login()

## Convert and Publish Shapefile to Triangular Mesh

In the cells below we:
1. Read the organization, hub, and workspace from the widget selections above
2. Specify the path to our shapefile
3. Add optional tags
4. Call `convert_shp` to convert and publish

The shapefile will be:
- Translated to vertices, triangles, and shapes which are stored in parquet files
- Formatted according to the triangle-mesh schema
- Published to your evo workspace

In [ ]:
# Install local package into this notebook kernel (run once per environment)
%pip install -e ../..

In [ ]:
from pathlib import Path

from evo.data_converters.shp.importer.shp_to_evo import convert_shp

# Get workspace from widget selections
service_mgr = manager._service_manager
org = service_mgr.get_current_organization()
hub = service_mgr.get_current_hub()
workspace = service_mgr.get_current_workspace()
selected_workspace_id = workspace.id

# Path to your shapefile (can be any of the individual files or just the extensionless name)
# Using relative path from notebook location
filename = "../../tests/data/test_shape.shp"

# PRJ File (Optional)
filename_prj = "../../tests/data/test_shape.prj"

# Tags to add to the geoscience object
tags = {"Source": "Jupyter Notebook", "Type": "Shapefile Triangular Mesh"}

# Upload path in Evo workspace
upload_path = "shapefile/imports"

print(f"Organization: {org.display_name}")
print(f"Hub: {hub.display_name}")
print(f"Workspace: {workspace.display_name} (ID: {selected_workspace_id})")
print(f"Shapefile: {Path(filename).name}")
print("\nConverting and publishing...")

# Convert and publish
results = convert_shp(
    filepath=filename,
    filepath_prj=filename_prj,
    tags=tags,
    evo_workspace_metadata=None,
    service_manager_widget=manager,
    upload_path=upload_path,
    publish_objects=True,
    overwrite_existing_objects=True,
)

# Print results
print("\n✅ Successfully published!")
print("\nPublished objects:")
for obj_metadata in results:
    print(f"  - {obj_metadata.name}")
    print(f"    ID: {obj_metadata.id}")

## Convert Without Publishing (for testing)

You can also convert the Shapefile without publishing to inspect the resulting object:

In [ ]:
import json

from evo.data_converters.shp.importer.shp_to_evo import convert_shp

# Convert but don't publish
shp_objects = convert_shp(
    filepath="../../tests/data/test_shape.shp",
    filepath_prj="../../tests/data/test_shape.prj",
    upload_path="./parquet_output",
    publish_objects=False,  # Don't publish
)

# Inspect the mesh object
shp = shp_objects[0]
print(f"Shape name: {shp.name}")

shp_dict = shp.as_dict()  # Use as_dict() instead of to_dict()
print("\nJSON representation:")
print(json.dumps(shp_dict, indent=2))